# Auditoría de coherencia metodológica del PAA

Este notebook verifica la coherencia entre preparación, clustering, adaptación de Nesterov y modelado supervisado. No redefine la metodología ni entrena modelos.

Objetivo común: caracterizar perfiles meteorológicos mediante clustering y clasificar `presencia_firms` a nivel departamento–semana. FIRMS representa detecciones o anomalías térmicas, no incendios confirmados.

In [1]:
from pathlib import Path
import json
import pandas as pd

RUTA_SEMANAL = Path("data/processed/firms_open_meteo_weekly/open_meteo_firms_weekly_target_available.parquet")
RUTA_MODELO = Path("data/processed/modelado/dataset_modelado_presencia_firms.parquet")
RUTA_NESTEROV = Path("data/processed/nesterov/nesterov_departamento_semana_open_meteo_2018_2025.parquet")
CLAVES = ["departamento", "fecha_inicio_semana"]
VARIABLES_CLUSTERING = [
    "temperature_2m_max_mean_weekly",
    "relative_humidity_2m_min_mean_weekly",
    "wind_speed_10m_max_mean_weekly",
    "precipitation_sum_weekly",
]

## 1. Integridad y alineación de los datos

In [2]:
semanal = pd.read_parquet(RUTA_SEMANAL)
modelo = pd.read_parquet(RUTA_MODELO)
nesterov = pd.read_parquet(RUTA_NESTEROV)
for tabla in (semanal, modelo, nesterov):
    tabla["fecha_inicio_semana"] = pd.to_datetime(tabla["fecha_inicio_semana"], errors="raise")

for nombre, tabla in {"semanal": semanal, "modelo": modelo, "nesterov": nesterov}.items():
    assert len(tabla) == 7923, f"{nombre}: número inesperado de filas"
    assert tabla["departamento"].nunique() == 19
    assert not tabla.duplicated(CLAVES).any()
    assert not tabla.isna().any().any(), f"{nombre}: hay faltantes"
    assert tabla.groupby("departamento").size().eq(417).all()

claves_semanal = semanal[CLAVES].sort_values(CLAVES).reset_index(drop=True)
assert claves_semanal.equals(modelo[CLAVES].sort_values(CLAVES).reset_index(drop=True))
assert claves_semanal.equals(nesterov[CLAVES].sort_values(CLAVES).reset_index(drop=True))
assert modelo["presencia_firms"].eq(modelo["cantidad_detecciones"].gt(0).astype("int8")).all()
assert modelo["perfil_meteorologico"].nunique() == 3

pd.DataFrame({
    "dataset": ["Fuente semanal", "Modelado", "Nesterov separado"],
    "filas": [len(semanal), len(modelo), len(nesterov)],
    "departamentos": [semanal.departamento.nunique(), modelo.departamento.nunique(), nesterov.departamento.nunique()],
    "inicio": [semanal.fecha_inicio_semana.min(), modelo.fecha_inicio_semana.min(), nesterov.fecha_inicio_semana.min()],
    "fin": [semanal.fecha_inicio_semana.max(), modelo.fecha_inicio_semana.max(), nesterov.fecha_inicio_semana.max()],
})

,dataset,filas,departamentos,inicio,fin
0,Fuente semanal,7923,19,2018-01-01,2025-12-22
1,Modelado,7923,19,2018-01-01,2025-12-22
2,Nesterov separado,7923,19,2018-01-01,2025-12-22


## 2. Roles de las variables y riesgo de fuga

La tabla distingue propósito analítico de disponibilidad técnica. La inclusión definitiva de contexto, perfiles y Nesterov en modelos supervisados queda pendiente de definición metodológica.

In [3]:
roles = pd.DataFrame([
    ["departamento", "Panel semanal", "No", "Candidata contextual", "No", "No; requiere codificación"],
    ["mes/estación", "Fecha de la semana", "No", "Candidatas contextuales", "No", "No"],
    ["temperatura", "Open-Meteo", "Sí", "Candidata", "No", "No"],
    ["humedad", "Open-Meteo", "Sí", "Candidata", "No", "No"],
    ["viento", "Open-Meteo", "Sí", "Candidata", "No", "No"],
    ["precipitación", "Open-Meteo", "Sí, con log1p", "Candidata", "No", "No"],
    ["perfil K-Means", "Clustering meteorológico", "Salida", "Candidata", "No", "Controlado si el fit usa solo train"],
    ["valor Nesterov", "Adaptación Open-Meteo/INUMET", "No", "Candidata futura", "No", "No; posible redundancia meteorológica"],
    ["nivel Nesterov", "Intervalos publicados por INUMET", "No", "Candidata futura", "No", "No; posible redundancia meteorológica"],
    ["cantidad_detecciones", "FIRMS", "No", "No", "Sí: define presencia_firms", "Sí si entra en X"],
    ["presencia_firms", "cantidad_detecciones > 0", "No", "Target", "Es el target", "Sí si entra en X"],
    ["lags FIRMS históricos", "FIRMS de semanas anteriores", "No", "Pendiente", "Sí, pero temporal", "Potencial: exigir pasado estricto"],
], columns=["Variable", "Origen", "En clustering", "En supervisado", "Derivada del target", "Riesgo de leakage"])
roles

,Variable,Origen,En clustering,En supervisado,Derivada del target,Riesgo de leakage
0,departamento,Panel semanal,No,Candidata contextual,No,No; requiere codificación
1,mes/estación,Fecha de la semana,No,Candidatas contextuales,No,No
2,temperatura,Open-Meteo,Sí,Candidata,No,No
3,humedad,Open-Meteo,Sí,Candidata,No,No
4,viento,Open-Meteo,Sí,Candidata,No,No
5,precipitación,Open-Meteo,"Sí, con log1p",Candidata,No,No
6,perfil K-Means,Clustering meteorológico,Salida,Candidata,No,Controlado si el fit usa solo train
7,valor Nesterov,Adaptación Open-Meteo/INUMET,No,Candidata futura,No,No; posible redundancia meteorológica
8,nivel Nesterov,Intervalos publicados por INUMET,No,Candidata futura,No,No; posible redundancia meteorológica
9,cantidad_detecciones,FIRMS,No,No,Sí: define presencia_firms,Sí si entra en X


## 3. Verificación del código declarado

In [4]:
def codigo_notebook(ruta):
    contenido = json.loads(Path(ruta).read_text(encoding="utf-8"))
    return "\n".join("".join(celda.get("source", [])) for celda in contenido["cells"] if celda["cell_type"] == "code")

codigo_preparacion = codigo_notebook("Preparacion_Dataset_Modelado.ipynb")
codigo_modelos = codigo_notebook("data/processed/modelado/Modelos.ipynb")

comprobaciones = {
    "K-Means usa k=3": "KMeans(n_clusters=3" in codigo_preparacion,
    "Scaler ajustado con train": "scaler.fit_transform(matriz_train)" in codigo_preparacion,
    "K-Means ajustado con train": "kmeans.fit(matriz_train_escalada)" in codigo_preparacion,
    "Target es cantidad > 0": '.gt(0).astype("int8")' in codigo_preparacion,
    "Dummy usa most_frequent": 'DummyClassifier(strategy="most_frequent")' in codigo_modelos,
    "No hay predicción de test en Modelos": "predict(X_test" not in codigo_modelos and "predict_proba(X_test" not in codigo_modelos,
    "Variables de clustering exactas": all(variable in codigo_preparacion for variable in VARIABLES_CLUSTERING),
    "FIRMS no figura en lista meteorológica": "cantidad_detecciones" not in VARIABLES_CLUSTERING and "presencia_firms" not in VARIABLES_CLUSTERING,
}
assert all(comprobaciones.values())
pd.Series(comprobaciones, name="cumple")

K-Means usa k=3                           True
Scaler ajustado con train                 True
K-Means ajustado con train                True
Target es cantidad > 0                    True
Dummy usa most_frequent                   True
No hay predicción de test en Modelos      True
Variables de clustering exactas           True
FIRMS no figura en lista meteorológica    True
Name: cumple, dtype: bool

## 4. Estado por componente

In [5]:
estado = pd.DataFrame([
    ["Calidad de datos", "Coherente", "Calidad_Preparacion_Datos: unidad, faltantes, duplicados y rangos", "Mantener Nesterov fuera de ese flujo"],
    ["K-Means", "Coherente", "Cuatro variables meteorológicas; k=3 principal", "k=4 solo sensibilidad"],
    ["DBSCAN", "Coherente como complemento", "No fuerza k y usa la misma matriz meteorológica", "No usar grupos diminutos como solución principal"],
    ["Nesterov", "Coherente con salvedad", "Reglas INUMET aplicadas sobre sustitutos Open-Meteo", "Llamarlo adaptación, no índice oficial"],
    ["Target FIRMS", "Coherente", "presencia_firms = (cantidad_detecciones > 0)", "Excluir cantidad y derivados contemporáneos de X"],
    ["Dataset supervisado", "Coherente", "7.923 claves y tres perfiles; fit del perfil solo en train", "Split definitivo pendiente"],
    ["Modelos", "Coherente", "Dummy ajustado en train y evaluado en validation", "Modelos finales y split definitivo pendientes"],
    ["MIRA/FIREDpy", "Exploratorios", "Fuentes externas fuera del pipeline principal", "No cambiar el target FIRMS vigente"],
], columns=["Componente", "Estado", "Evidencia", "Acción o límite"])
estado

,Componente,Estado,Evidencia,Acción o límite
0,Calidad de datos,Coherente,"Calidad_Preparacion_Datos: unidad, faltantes, ...",Mantener Nesterov fuera de ese flujo
1,K-Means,Coherente,Cuatro variables meteorológicas; k=3 principal,k=4 solo sensibilidad
2,DBSCAN,Coherente como complemento,No fuerza k y usa la misma matriz meteorológica,No usar grupos diminutos como solución principal
3,Nesterov,Coherente con salvedad,Reglas INUMET aplicadas sobre sustitutos Open-...,"Llamarlo adaptación, no índice oficial"
4,Target FIRMS,Coherente,presencia_firms = (cantidad_detecciones > 0),Excluir cantidad y derivados contemporáneos de X
5,Dataset supervisado,Coherente,7.923 claves y tres perfiles; fit del perfil s...,Split definitivo pendiente
6,Modelos,Coherente,Dummy ajustado en train y evaluado en validation,Modelos finales y split definitivo pendientes
7,MIRA/FIREDpy,Exploratorios,Fuentes externas fuera del pipeline principal,No cambiar el target FIRMS vigente


## 5. Conclusión

Las tres piezas quedan separadas: los perfiles se obtienen exclusivamente de cuatro variables meteorológicas; Nesterov es una adaptación meteorológica independiente construida con Open-Meteo; y `presencia_firms` es el target supervisado. `cantidad_detecciones` no es predictor.

La decisión sobre el split supervisado definitivo, la codificación de variables nominales y la incorporación de Nesterov o lags históricos continúa abierta. Los notebooks MIRA/FIREDpy son exploratorios y no forman parte del flujo principal.